[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/37.%20The%20Equipment%20Selection%20Problem%20%28RTG%20vs.%20RMG%20vs.%20Straddle%29/P37-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 37. The Equipment Selection Problem (RTG vs. RMG vs. Straddle)
## Tier 7: The Human-AI Symbiotic Partnership (The Centaur Model)

### Goal

This notebook illustrates a human-AI partnership for strategic equipment selection at a container terminal. A rule-based recommendation component proposes equipment configurations, a simulated domain expert provides contextual feedback, and transparent rules combine the two into a collaborative decision. The example emphasizes how to represent the interaction, record its outcomes, and track illustrative trust and satisfaction metrics.

### Inputs and outputs

The inputs are equipment attributes, a small set of terminal scenarios, and profiles for four simulated experts. The notebook produces AI recommendations, human feedback, collaborative configurations, calculated outcome metrics, and visual summaries of the simulated interactions.

### Method and interpretation

The method is a staged, rule-based simulation: generate a recommendation, obtain expert feedback, apply the corresponding decision rules, calculate metrics, and update trust. These steps illustrate a human-AI workflow; they do not constitute a trained AI model, an optimization algorithm, or empirical evidence that collaboration outperforms either party working alone.

The recommendations and expert feedback are generated by explicit rules in the code. They are not predictions from operational data. The confidence scores and trust and satisfaction scores are illustrative model quantities, not statistically estimated probabilities or validated measures.

### Model assumptions and scope

- The three equipment types are represented by fixed attributes. Capital costs are in millions of dollars, and productivity is in containers per hour per unit.
- A configuration is a mapping from equipment type to a nonnegative integer unit count. Total productivity is modeled as the sum of the individual equipment productivity values; interactions between equipment units are not represented.
- The recommendation rules use budget and TEU volume. The scenario's peak moves and truck percentage are available to the simulated human feedback rules but do not enter the recommendation rules unless explicitly used there.
- The budget is used to construct an initial configuration, but the collaborative adjustments are not constrained by a budget-feasibility rule. The reported budget utilization is therefore a diagnostic, not a guarantee of budget feasibility.
- Capital cost is used in the cost-efficiency calculation. Operating cost, maintenance, footprint, energy consumption, financing, infrastructure, and lifecycle effects are not included in that calculation.
- Feedback, satisfaction, and trust updates are simplified deterministic rules. No real experts, operational outcomes, external data, or learning from feedback are involved.

### What to look for

- How the recommendation rules respond to TEU volume and budget.
- How particular phrases in simulated expert feedback trigger configuration adjustments.
- How total capital cost, total hourly productivity, cost efficiency, budget utilization, satisfaction, and trust are calculated.
- Why these simulated metrics describe the rules in this example rather than proving that one decision process is superior.

### Illustrative context

A terminal may combine equipment with different cost, productivity, and flexibility characteristics. An expert might also consider future shipping alliances, infrastructure needs, or implementation risk. The notebook represents this context through transparent scenario rules; it does not claim to analyze a real Mediterranean terminal or to establish a specific equipment mix.

### When this modeling pattern is useful

A structured human-AI workflow can make assumptions and decision adjustments visible when a strategic choice includes both quantitative inputs and contextual judgment. Its limitations include dependence on the quality of the rules, the availability and judgment of experts, possible human bias, and the absence of empirical validation. A real deployment would need suitable data, governance, and evaluation before operational use.

In [ ]:
import random
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)

print('Equipment Selection: Human-AI Symbiotic Partnership')
print('The Centaur Model: Rule-Based Recommendations with Human Oversight')

### Human-AI Partnership Data Structure

The data classes below give names and types to the information passed between stages. For example, an `AIRecommendation` keeps the proposed configuration together with its confidence score, reasoning, alternatives, and stated uncertainty factors. This makes the simulated decision process easier to inspect than a collection of unrelated values.

In [ ]:
@dataclass
class EquipmentType:
    name: str
    capital_cost: float
    operating_cost: float
    productivity: float
    flexibility: float
    maintenance: float
    footprint: float
    energy_consumption: float


@dataclass
class AIRecommendation:
    equipment_config: Dict[str, int]
    confidence_score: float
    reasoning: List[str]
    alternatives: List[Dict[str, int]]
    data_sources: List[str]
    uncertainty_factors: List[str]


@dataclass
class HumanExpert:
    expert_id: str
    name: str
    expertise_area: str
    experience_years: int
    decision_style: str
    trust_level: float
    satisfaction_score: float


@dataclass
class PartnershipInteraction:
    interaction_id: str
    timestamp: datetime
    scenario_description: str
    ai_recommendation: AIRecommendation
    human_feedback: str
    final_decision: Dict[str, int]
    outcome_metrics: Dict[str, float]
    satisfaction_rating: float
    trust_change: float


@dataclass
class TrustMetrics:
    competence: float
    integrity: float
    benevolence: float
    predictability: float
    overall_trust: float


# The attributes are illustrative model inputs, not measurements from a terminal.
equipment_types = {
    'RTG': EquipmentType(
        name='Rubber-Tired Gantry',
        capital_cost=2.5,
        operating_cost=0.8,
        productivity=25,
        flexibility=0.8,
        maintenance=0.3,
        footprint=200,
        energy_consumption=3.2,
    ),
    'RMG': EquipmentType(
        name='Rail-Mounted Gantry',
        capital_cost=3.2,
        operating_cost=0.6,
        productivity=35,
        flexibility=0.4,
        maintenance=0.25,
        footprint=150,
        energy_consumption=2.8,
    ),
    'Straddle': EquipmentType(
        name='Straddle Carrier',
        capital_cost=4.1,
        operating_cost=1.1,
        productivity=20,
        flexibility=0.9,
        maintenance=0.4,
        footprint=100,
        energy_consumption=4.5,
    ),
}

print('Illustrative equipment attributes:')
for equipment_key, equipment in equipment_types.items():
    print(f'{equipment_key}: {equipment.name}')
    print(f'  Capital cost: ${equipment.capital_cost:.1f} million per unit')
    print(f'  Productivity: {equipment.productivity} containers per hour per unit')
    print(f'  Flexibility score: {equipment.flexibility:.1f}')
    print()

### Human-AI Symbiotic Partnership System

The partnership class implements the notebook's main sequence. It first generates a rule-based recommendation, then maps simulated expert feedback to explicit configuration changes. It calculates outcome metrics from the final unit counts and equipment attributes, and updates the illustrative trust measures.

For a configuration \(x\), the model calculates capital cost and nominal hourly productivity as

$$
C(x) = \sum_{e \in E} x_e c_e,
\qquad
P(x) = \sum_{e \in E} x_e p_e,
\qquad
Q(x) = \begin{cases}
P(x) / C(x), & C(x) > 0, \\
0, & C(x) = 0.
\end{cases}
$$

Here, \(E\) is the set of equipment types, \(x_e\) is the number of units of type \(e\), \(c_e\) is its capital cost in millions of dollars per unit, and \(p_e\) is its productivity in containers per hour per unit. Thus \(Q(x)\) has units of containers per hour per million dollars of capital cost. This is a simplified capital-cost efficiency measure, not a lifecycle cost measure.

Trust is represented by four components. After each interaction, the code updates those components using the same trust increment and combines them with fixed weights:

$$
T = 0.30 T_{\mathrm{competence}} + 0.25 T_{\mathrm{integrity}} + 0.20 T_{\mathrm{benevolence}} + 0.25 T_{\mathrm{predictability}}.
$$

Each component is bounded between zero and one. The class records the resulting overall trust after every interaction so that the plotted history reflects the component updates.

In [ ]:
class HumanAISymbioticPartnership:
    def __init__(self, equipment_types: Dict[str, EquipmentType]):
        self.equipment_types = equipment_types
        self.human_experts: List[HumanExpert] = []
        self.partnership_history: List[PartnershipInteraction] = []
        self.current_trust_metrics = TrustMetrics(0.5, 0.5, 0.5, 0.5, 0.5)
        self.trust_history = [self.current_trust_metrics.overall_trust]

    def add_human_expert(self, expert: HumanExpert) -> None:
        """Register an expert whose feedback can be simulated."""
        self.human_experts.append(expert)

    def generate_ai_recommendation(self, scenario: Dict) -> AIRecommendation:
        """Create an illustrative recommendation using explicit scenario rules."""
        budget = scenario.get('budget', 200)
        teu_volume = scenario.get('teu_volume', 2.5)

        if teu_volume > 3.0:
            primary_config = {
                'RMG': int(budget / 3.2 * 0.7),
                'RTG': int(budget / 2.5 * 0.2),
                'Straddle': int(budget / 4.1 * 0.1),
            }
            confidence = 0.85
            reasoning = [
                'The scenario volume rule favors RMG productivity at this scale.',
                'The recommendation rule assigns the largest budget share to RMG.',
                'The stated confidence is an illustrative rule output, not a calibrated probability.',
            ]
        elif teu_volume < 2.0:
            primary_config = {
                'RTG': int(budget / 2.5 * 0.6),
                'RMG': int(budget / 3.2 * 0.1),
                'Straddle': int(budget / 4.1 * 0.3),
            }
            confidence = 0.78
            reasoning = [
                'The scenario volume rule favors RTG flexibility at this scale.',
                'The recommendation rule assigns the largest budget share to RTG.',
                'The stated confidence is an illustrative rule output, not a calibrated probability.',
            ]
        else:
            primary_config = {
                'RMG': int(budget / 3.2 * 0.4),
                'RTG': int(budget / 2.5 * 0.3),
                'Straddle': int(budget / 4.1 * 0.3),
            }
            confidence = 0.72
            reasoning = [
                'The scenario volume rule uses a balanced equipment mix.',
                'The recommendation rule divides its budget shares across all three types.',
                'The stated confidence is an illustrative rule output, not a calibrated probability.',
            ]

        alternatives = [
            {
                'RTG': int(budget / 2.5 * 0.8),
                'RMG': 0,
                'Straddle': int(budget / 4.1 * 0.2),
            },
            {
                'RTG': 0,
                'RMG': int(budget / 3.2 * 0.8),
                'Straddle': int(budget / 4.1 * 0.2),
            },
            {
                'RTG': int(budget / 2.5 * 0.2),
                'RMG': int(budget / 3.2 * 0.1),
                'Straddle': int(budget / 4.1 * 0.7),
            },
        ]

        data_sources = [
            'Illustrative equipment attributes defined in this notebook',
            'Scenario budget and TEU volume supplied to the rule-based recommender',
        ]
        uncertainty_factors = [
            'Future shipping alliance changes',
            'Regulatory environment evolution',
            'Technology advancement uncertainty',
            'Market demand volatility',
        ]

        return AIRecommendation(
            equipment_config=primary_config,
            confidence_score=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            data_sources=data_sources,
            uncertainty_factors=uncertainty_factors,
        )

    def simulate_human_expert_feedback(
        self,
        expert: HumanExpert,
        ai_recommendation: AIRecommendation,
        scenario: Dict,
    ) -> str:
        """Return rule-generated feedback for an expert profile and scenario."""
        teu_volume = scenario.get('teu_volume', 2.5)
        truck_pct = scenario.get('truck_percentage', 75)

        if expert.expertise_area == 'Operations':
            if expert.decision_style == 'conservative':
                feedback = (
                    'Concerned about RMG maintenance complexity. Prefer more RTG '
                    'units for operational flexibility and easier maintenance.'
                )
            elif expert.decision_style == 'aggressive':
                feedback = (
                    'Recommend increasing RMG proportion for higher efficiency. '
                    'Automation potential outweighs maintenance concerns.'
                )
            else:
                feedback = (
                    'The AI recommendation is reasonable, but consider increasing '
                    'the straddle carrier proportion for landside operations.'
                )
        elif expert.expertise_area == 'Finance':
            if expert.decision_style == 'conservative':
                feedback = (
                    'Capital expenditure may be too high. Recommend phased '
                    'implementation starting with RTG units.'
                )
            elif expert.decision_style == 'aggressive':
                feedback = (
                    'Investment may be justified by long-term efficiency gains. '
                    'Recommend full implementation as proposed.'
                )
            else:
                feedback = (
                    'Consider total cost of ownership over 10 years. RMG may have '
                    'higher upfront but lower operating costs.'
                )
        elif expert.expertise_area == 'Strategy':
            if teu_volume > 2.5 and truck_pct > 70:
                feedback = (
                    'Potential shipping alliance changes may increase feeder vessel '
                    'traffic. Consider more flexible equipment and a larger RTG share.'
                )
            else:
                feedback = (
                    'Consider future growth scenarios and competitive positioning '
                    'alongside the AI recommendation.'
                )
        elif expert.expertise_area == 'Engineering':
            feedback = (
                'Infrastructure requirements for RMG may be underestimated. '
                'Consider additional rail investment and foundation work.'
            )
        else:
            feedback = (
                'Consider safety, automation, and operator training requirements '
                when assessing the equipment configuration.'
            )

        return feedback

    def collaborative_decision_making(
        self,
        scenario: Dict,
        selected_expert: HumanExpert,
    ) -> PartnershipInteraction:
        """Apply feedback rules, calculate metrics, and record one interaction."""
        ai_recommendation = self.generate_ai_recommendation(scenario)
        human_feedback = self.simulate_human_expert_feedback(
            selected_expert,
            ai_recommendation,
            scenario,
        )

        # Copy the proposed configuration so adjustments do not alter the recommendation.
        final_decision = ai_recommendation.equipment_config.copy()

        if 'flexibility' in human_feedback.lower() or 'RTG' in human_feedback:
            adjustment = int(final_decision.get('RMG', 0) * 0.3)
            final_decision['RTG'] = final_decision.get('RTG', 0) + adjustment
            final_decision['RMG'] = max(
                0,
                final_decision.get('RMG', 0) - adjustment,
            )

        if 'phased' in human_feedback.lower() or 'conservative' in human_feedback.lower():
            for equipment_key in final_decision:
                final_decision[equipment_key] = int(
                    final_decision[equipment_key] * 0.8
                )

        if 'straddle' in human_feedback.lower():
            adjustment = int(final_decision.get('RTG', 0) * 0.2)
            final_decision['Straddle'] = (
                final_decision.get('Straddle', 0) + adjustment
            )
            final_decision['RTG'] = max(
                0,
                final_decision.get('RTG', 0) - adjustment,
            )

        total_cost = sum(
            final_decision.get(key, 0) * self.equipment_types[key].capital_cost
            for key in self.equipment_types
        )
        total_productivity = sum(
            final_decision.get(key, 0) * self.equipment_types[key].productivity
            for key in self.equipment_types
        )
        budget = scenario.get('budget', 200)
        outcome_metrics = {
            'total_cost': total_cost,
            'total_productivity': total_productivity,
            'cost_efficiency': (
                total_productivity / total_cost if total_cost > 0 else 0
            ),
            'budget_utilization': total_cost / budget if budget > 0 else 0,
            'implementation_complexity': self._calculate_implementation_complexity(
                final_decision
            ),
        }

        satisfaction_rating = self._calculate_satisfaction(
            human_feedback,
            final_decision,
            ai_recommendation,
        )
        trust_change = self._update_trust_metrics(
            ai_recommendation,
            human_feedback,
            outcome_metrics,
        )
        selected_expert.trust_level = max(
            0,
            min(1, selected_expert.trust_level + trust_change * 0.1),
        )
        selected_expert.satisfaction_score = satisfaction_rating

        interaction = PartnershipInteraction(
            interaction_id=f'INT_{len(self.partnership_history) + 1:03d}',
            timestamp=datetime.now(),
            scenario_description=scenario.get(
                'description',
                'Equipment selection scenario',
            ),
            ai_recommendation=ai_recommendation,
            human_feedback=human_feedback,
            final_decision=final_decision,
            outcome_metrics=outcome_metrics,
            satisfaction_rating=satisfaction_rating,
            trust_change=trust_change,
        )
        self.partnership_history.append(interaction)
        return interaction

    def _calculate_implementation_complexity(
        self,
        decision: Dict[str, int],
    ) -> float:
        """Calculate a bounded illustrative complexity score."""
        complexity = (
            decision.get('RMG', 0) * 0.8
            + decision.get('RTG', 0) * 0.5
            + decision.get('Straddle', 0) * 0.6
        )
        return min(1.0, complexity / 50)

    def _calculate_satisfaction(
        self,
        feedback: str,
        decision: Dict[str, int],
        ai_recommendation: AIRecommendation,
    ) -> float:
        """Score whether the simulated adjustment addresses stated concerns."""
        base_satisfaction = 0.7

        if (
            'flexibility' in feedback.lower()
            and decision.get('RTG', 0)
            > ai_recommendation.equipment_config.get('RTG', 0)
        ):
            base_satisfaction += 0.15

        if 'phased' in feedback.lower() or 'conservative' in feedback.lower():
            total_units = sum(decision.values())
            ai_units = sum(ai_recommendation.equipment_config.values())
            if total_units < ai_units:
                base_satisfaction += 0.1

        if (
            'straddle' in feedback.lower()
            and decision.get('Straddle', 0)
            > ai_recommendation.equipment_config.get('Straddle', 0)
        ):
            base_satisfaction += 0.1

        return min(1.0, base_satisfaction)

    def _update_trust_metrics(
        self,
        ai_recommendation: AIRecommendation,
        human_feedback: str,
        outcome_metrics: Dict[str, float],
    ) -> float:
        """Apply the example's deterministic trust-update rules."""
        trust_change = 0.0

        if outcome_metrics['cost_efficiency'] > 15:
            trust_change += 0.1

        if outcome_metrics['budget_utilization'] < 0.95:
            trust_change += 0.05

        if (
            ai_recommendation.confidence_score > 0.8
            and outcome_metrics['cost_efficiency'] > 12
        ):
            trust_change += 0.08

        metrics = self.current_trust_metrics
        metrics.competence = max(
            0,
            min(1, metrics.competence + trust_change),
        )
        metrics.integrity = max(
            0,
            min(1, metrics.integrity + trust_change * 0.8),
        )
        metrics.benevolence = max(
            0,
            min(1, metrics.benevolence + trust_change * 0.6),
        )
        metrics.predictability = max(
            0,
            min(1, metrics.predictability + trust_change * 0.7),
        )
        metrics.overall_trust = (
            metrics.competence * 0.30
            + metrics.integrity * 0.25
            + metrics.benevolence * 0.20
            + metrics.predictability * 0.25
        )
        self.trust_history.append(metrics.overall_trust)
        return trust_change

    def get_partnership_analytics(self) -> Dict:
        """Summarize recorded interactions and their illustrative metrics."""
        if not self.partnership_history:
            return {}

        total_interactions = len(self.partnership_history)
        avg_satisfaction = np.mean(
            [item.satisfaction_rating for item in self.partnership_history]
        )
        agreement_rate = sum(
            1
            for item in self.partnership_history
            if 'recommendation is reasonable' in item.human_feedback.lower()
            or 'supports the AI recommendation' in item.human_feedback.lower()
        ) / total_interactions
        avg_outcome_quality = np.mean(
            [
                item.outcome_metrics['cost_efficiency']
                for item in self.partnership_history
            ]
        )

        expert_performance = {}
        for expert in self.human_experts:
            expert_interactions = [
                item
                for item in self.partnership_history
                if item.scenario_description.startswith(expert.expertise_area)
            ]
            if expert_interactions:
                expert_performance[expert.expertise_area] = {
                    'interactions': len(expert_interactions),
                    'avg_satisfaction': np.mean(
                        [item.satisfaction_rating for item in expert_interactions]
                    ),
                    'trust_level': expert.trust_level,
                }

        return {
            'total_interactions': total_interactions,
            'avg_satisfaction': avg_satisfaction,
            'agreement_rate': agreement_rate,
            'avg_outcome_quality': avg_outcome_quality,
            'trust_evolution': self.trust_history.copy(),
            'current_trust': self.current_trust_metrics.overall_trust,
            'expert_performance': expert_performance,
        }

print('Human-AI partnership rules and analytics are ready.')

### Initialize Human Experts and Partnership

Each expert profile supplies an expertise area and decision style used by the feedback rules. These profiles are inputs to a simulation; they do not represent observed behavior by real people.

In [ ]:
experts = [
    HumanExpert(
        expert_id='EXP001',
        name='Dr. Sarah Chen',
        expertise_area='Operations',
        experience_years=15,
        decision_style='balanced',
        trust_level=0.5,
        satisfaction_score=0.5,
    ),
    HumanExpert(
        expert_id='EXP002',
        name='Mark Rodriguez',
        expertise_area='Finance',
        experience_years=12,
        decision_style='conservative',
        trust_level=0.5,
        satisfaction_score=0.5,
    ),
    HumanExpert(
        expert_id='EXP003',
        name='Prof. James Liu',
        expertise_area='Strategy',
        experience_years=20,
        decision_style='conservative',
        trust_level=0.5,
        satisfaction_score=0.5,
    ),
    HumanExpert(
        expert_id='EXP004',
        name='Amanda Foster',
        expertise_area='Engineering',
        experience_years=8,
        decision_style='aggressive',
        trust_level=0.5,
        satisfaction_score=0.5,
    ),
]

partnership = HumanAISymbioticPartnership(equipment_types)
for expert in experts:
    partnership.add_human_expert(expert)

print('=== HUMAN-AI PARTNERSHIP INITIALIZED ===\n')
print(f'Number of simulated experts: {len(experts)}')
print(f'Initial overall trust: {partnership.current_trust_metrics.overall_trust:.2f}\n')
print('Expert profiles:')
for expert in experts:
    print(
        f'{expert.name} — {expert.expertise_area} '
        f'({expert.experience_years} years of stated experience)'
    )
    print(f'  Decision style: {expert.decision_style}')
    print(f'  Initial individual trust score: {expert.trust_level:.2f}')
    print()

### Run Collaborative Decision-Making Scenarios

The scenarios provide the inputs for the recommendation and feedback rules. For each scenario, the notebook selects the expert whose expertise area matches the scenario, records the recommendation and feedback, and displays the resulting configuration and metrics. Integer conversion in the recommendation rules means the resulting unit counts are whole numbers.

In [ ]:
scenarios = [
    {
        'description': 'Operations - Peak Season Optimization',
        'expertise_area': 'Operations',
        'budget': 200,
        'teu_volume': 3.2,
        'peak_moves': 4500,
        'truck_percentage': 75,
    },
    {
        'description': 'Finance - Budget Constraint Scenario',
        'expertise_area': 'Finance',
        'budget': 150,
        'teu_volume': 2.5,
        'peak_moves': 3000,
        'truck_percentage': 70,
    },
    {
        'description': 'Strategy - Market Expansion Planning',
        'expertise_area': 'Strategy',
        'budget': 250,
        'teu_volume': 2.8,
        'peak_moves': 3500,
        'truck_percentage': 65,
    },
    {
        'description': 'Engineering - Infrastructure Upgrade',
        'expertise_area': 'Engineering',
        'budget': 220,
        'teu_volume': 2.7,
        'peak_moves': 3400,
        'truck_percentage': 72,
    },
]

print('=== COLLABORATIVE DECISION-MAKING SCENARIOS ===\n')
interactions = []

for scenario in scenarios:
    selected_expert = next(
        expert
        for expert in experts
        if expert.expertise_area == scenario['expertise_area']
    )

    print(f'--- {scenario["description"]} ---')
    print(f'Expert profile: {selected_expert.name}')
    print(
        f'Budget: ${scenario["budget"]} million; '
        f'TEU volume: {scenario["teu_volume"]} million\n'
    )

    interaction = partnership.collaborative_decision_making(
        scenario,
        selected_expert,
    )
    interactions.append(interaction)

    print('Rule-based recommendation:')
    print(f'  Configuration: {interaction.ai_recommendation.equipment_config}')
    print(
        f'  Illustrative confidence score: '
        f'{interaction.ai_recommendation.confidence_score:.2f}'
    )
    print(f'  Rationale: {interaction.ai_recommendation.reasoning[0]}\n')

    print('Simulated expert feedback:')
    print(f'  {interaction.human_feedback}\n')

    print('Collaborative configuration and calculated metrics:')
    print(f'  Configuration: {interaction.final_decision}')
    print(
        f'  Capital cost: '
        f'${interaction.outcome_metrics["total_cost"]:.1f} million'
    )
    print(
        f'  Nominal productivity: '
        f'{interaction.outcome_metrics["total_productivity"]:.0f} containers/hour'
    )
    print(
        f'  Capital-cost efficiency: '
        f'{interaction.outcome_metrics["cost_efficiency"]:.2f} '
        f'containers/hour per $1 million'
    )
    print(
        f'  Budget utilization diagnostic: '
        f'{interaction.outcome_metrics["budget_utilization"]:.1%}'
    )
    print(f'  Illustrative satisfaction score: {interaction.satisfaction_rating:.2f}')
    print(f'  Trust increment used by the rules: {interaction.trust_change:+.3f}\n')
    print('-' * 60)
    print()

### Partnership Analytics and Performance Analysis

The analytics summarize the interactions already recorded. The agreement rate is a simple text-pattern indicator: it counts feedback containing one of the specified agreement phrases. It should not be interpreted as a validated measure of human acceptance. Trust evolution is read from the trust-component updates made during the interactions, rather than reconstructed from a separate approximation.

In [ ]:
analytics = partnership.get_partnership_analytics()

print('=== PARTNERSHIP ANALYTICS ===\n')
print(f'Interactions recorded: {analytics["total_interactions"]}')
print(f'Mean illustrative satisfaction: {analytics["avg_satisfaction"]:.3f}')
print(f'Agreement phrase rate: {analytics["agreement_rate"]:.1%}')
print(
    f'Mean capital-cost efficiency: '
    f'{analytics["avg_outcome_quality"]:.2f} containers/hour per $1 million'
)
print(f'Current overall trust score: {analytics["current_trust"]:.3f}\n')

print('Expert-profile summaries:')
for area, performance in analytics['expert_performance'].items():
    print(f'{area}:')
    print(f'  Interactions: {performance["interactions"]}')
    print(f'  Mean illustrative satisfaction: {performance["avg_satisfaction"]:.3f}')
    print(f'  Individual trust score: {performance["trust_level"]:.3f}\n')

trust_history = analytics['trust_evolution']
print('Overall trust history:')
for interaction_number, trust_score in enumerate(trust_history):
    print(f'  After interaction {interaction_number}: {trust_score:.3f}')

print(
    f'\nChange in overall trust score: '
    f'{trust_history[-1] - trust_history[0]:+.3f}'
)

### Comprehensive Visualization

The panels show the simulated trust trajectory, expert-profile satisfaction and trust scores, calculated recommendation-versus-collaboration efficiency, and interaction-level efficiency alongside satisfaction. The efficiency comparison uses configurations generated by the notebook's rules; it is not a comparison against independently measured human-only or AI-only outcomes.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    2,
    2,
    figsize=(16, 12),
)
fig.suptitle(
    'Illustrative Human-AI Partnership Analysis',
    fontsize=16,
    fontweight='bold',
)

# Trust scores are the overall values recorded after each rule-based update.
interaction_numbers = range(len(trust_history))
ax1.plot(
    interaction_numbers,
    trust_history,
    'o-',
    linewidth=2,
    markersize=8,
    color='blue',
    label='Overall trust score',
)
ax1.set_xlabel('Interaction number')
ax1.set_ylabel('Overall trust score (0–1)')
ax1.set_title('Trust Score Across Simulated Interactions')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)
ax1.legend()

for interaction_number, trust_score in zip(interaction_numbers, trust_history):
    if interaction_number > 0:
        scenario_name = interactions[
            interaction_number - 1
        ].scenario_description.split(' - ')[1].split(' ')[0]
        ax1.annotate(
            scenario_name,
            (interaction_number, trust_score),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8,
        )

# These are the final simulated profile scores after the scenarios.
expert_names = [expert.name.split(' ')[-1] for expert in experts]
satisfaction_scores = [expert.satisfaction_score for expert in experts]
trust_levels = [expert.trust_level for expert in experts]
x = np.arange(len(expert_names))
width = 0.35

bars_satisfaction = ax2.bar(
    x - width / 2,
    satisfaction_scores,
    width,
    label='Satisfaction score',
    alpha=0.7,
    color='green',
)
bars_trust = ax2.bar(
    x + width / 2,
    trust_levels,
    width,
    label='Individual trust score',
    alpha=0.7,
    color='blue',
)
ax2.set_xlabel('Expert profile')
ax2.set_ylabel('Score (0–1)')
ax2.set_title('Simulated Satisfaction and Individual Trust')
ax2.set_xticks(x)
ax2.set_xticklabels(expert_names, rotation=45, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

for bar_group in [bars_satisfaction, bars_trust]:
    for bar in bar_group:
        height = bar.get_height()
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f'{height:.2f}',
            ha='center',
            va='bottom',
            fontsize=8,
        )

# Compute recommendation efficiency from the same equipment attributes used
# for the collaborative configuration; no separate outcome is fabricated.
ai_efficiencies = []
collaborative_efficiencies = []
for interaction in interactions:
    ai_config = interaction.ai_recommendation.equipment_config
    ai_cost = sum(
        ai_config.get(key, 0) * equipment_types[key].capital_cost
        for key in equipment_types
    )
    ai_productivity = sum(
        ai_config.get(key, 0) * equipment_types[key].productivity
        for key in equipment_types
    )
    ai_efficiencies.append(
        ai_productivity / ai_cost if ai_cost > 0 else 0
    )
    collaborative_efficiencies.append(
        interaction.outcome_metrics['cost_efficiency']
    )

scenario_labels = [f'Scenario {index + 1}' for index in range(len(interactions))]
bar_positions = np.arange(len(scenario_labels))
comparison_width = 0.35
ax3.bar(
    bar_positions - comparison_width / 2,
    ai_efficiencies,
    comparison_width,
    label='Recommendation',
    color='steelblue',
    alpha=0.8,
)
ax3.bar(
    bar_positions + comparison_width / 2,
    collaborative_efficiencies,
    comparison_width,
    label='Collaborative configuration',
    color='darkorange',
    alpha=0.8,
)
ax3.set_xticks(bar_positions)
ax3.set_xticklabels(scenario_labels)
ax3.set_xlabel('Scenario')
ax3.set_ylabel('Containers/hour per $1 million of capital cost')
ax3.set_title('Calculated Capital-Cost Efficiency by Configuration')
ax3.legend()
ax3.grid(True, axis='y', alpha=0.3)

# Efficiency and satisfaction use separate vertical axes because their units differ.
interaction_labels = [f'Interaction {index + 1}' for index in range(len(interactions))]
efficiency_scores = [
    interaction.outcome_metrics['cost_efficiency']
    for interaction in interactions
]
interaction_satisfaction = [
    interaction.satisfaction_rating
    for interaction in interactions
]
ax4_twin = ax4.twinx()
line_efficiency = ax4.plot(
    interaction_labels,
    efficiency_scores,
    'o-',
    linewidth=2,
    markersize=6,
    color='blue',
    label='Capital-cost efficiency',
)
ax4.set_ylabel(
    'Containers/hour per $1 million of capital cost',
    color='blue',
)
ax4.tick_params(axis='y', labelcolor='blue')
ax4.set_xlabel('Interaction')
ax4.set_title('Calculated Efficiency and Simulated Satisfaction')
ax4.grid(True, alpha=0.3)
line_satisfaction = ax4_twin.plot(
    interaction_labels,
    interaction_satisfaction,
    's--',
    linewidth=2,
    markersize=6,
    color='red',
    label='Satisfaction score',
)
ax4_twin.set_ylabel('Satisfaction score (0–1)', color='red')
ax4_twin.tick_params(axis='y', labelcolor='red')
lines = line_efficiency + line_satisfaction
ax4.legend(lines, [line.get_label() for line in lines], loc='best')

fig.tight_layout()
plt.show()

print('Illustrative partnership visualization displayed.')

### Learning and Adaptation Analysis

This section examines how human feedback changes each configuration. In this notebook, the recommendation rules do not update after feedback: there is no trained model, stored preference model, or parameter-learning step. Therefore, the measured changes are human-driven adjustments within individual interactions, not evidence that the AI system learned or adapted across scenarios.

In [ ]:
print('=== HUMAN-DRIVEN CONFIGURATION ADJUSTMENTS ===\n')

for interaction_number, interaction in enumerate(interactions, start=1):
    ai_config = interaction.ai_recommendation.equipment_config
    final_config = interaction.final_decision
    scenario_name = interaction.scenario_description.split(' - ')[1]

    print(f'Interaction {interaction_number} — {scenario_name}:')
    print(f'  Rule-based recommendation: {ai_config}')
    print(f'  Collaborative configuration: {final_config}')

    changes = []
    for equipment_key in ['RTG', 'RMG', 'Straddle']:
        ai_count = ai_config.get(equipment_key, 0)
        final_count = final_config.get(equipment_key, 0)
        if ai_count != final_count:
            changes.append(f'{equipment_key}: {final_count - ai_count:+d} units')

    if changes:
        print(f'  Rule-generated adjustments: {", ".join(changes)}')
        print(f'  Feedback basis: {interaction.human_feedback}')
    else:
        print('  No configuration adjustment was triggered by the feedback rules.')

    print()

total_ai_units = sum(
    sum(interaction.ai_recommendation.equipment_config.values())
    for interaction in interactions
)
total_final_units = sum(
    sum(interaction.final_decision.values())
    for interaction in interactions
)

print('=== CONFIGURATION CHANGE SUMMARY ===')
print(f'Total units in recommendations: {total_ai_units}')
print(f'Total units in collaborative configurations: {total_final_units}')
if total_ai_units > 0:
    adjustment_rate = (total_final_units - total_ai_units) / total_ai_units * 100
    print(f'Net unit-count change relative to recommendations: {adjustment_rate:+.1f}%')
else:
    print('Net unit-count change is undefined because the recommendations contain no units.')

print('\nInterpretation:')
print(
    'The differences show the effect of the explicit feedback rules on these '
    'configurations. They do not measure model learning or establish which '
    'decision process performs better.'
)

### Results Summary and Recommendations

The final summary reports values computed from the simulated interactions. It distinguishes those values from claims that would require external evidence, such as measured human-only performance, validated AI-only performance, or demonstrated improvement from collaboration.

In [ ]:
print('=== HUMAN-AI PARTNERSHIP: SIMULATION SUMMARY ===\n')
print('Problem: Illustrative equipment selection with simulated human oversight')
print('Method: Rule-based recommendation, feedback, adjustment, and trust updates\n')

print('PARTNERSHIP SETUP:')
print(f'  Simulated expert profiles: {len(experts)}')
print('  Expertise areas: Operations, Finance, Strategy, Engineering')
print(f'  Scenarios processed: {len(interactions)}')
print(f'  Initial overall trust score: {trust_history[0]:.3f}')
print(f'  Final overall trust score: {trust_history[-1]:.3f}\n')

print('CALCULATED SIMULATION METRICS:')
print(f'  Mean illustrative satisfaction: {analytics["avg_satisfaction"]:.3f}')
print(f'  Agreement phrase rate: {analytics["agreement_rate"]:.1%}')
print(
    f'  Mean capital-cost efficiency: '
    f'{analytics["avg_outcome_quality"]:.2f} containers/hour per $1 million'
)
print(
    f'  Change in overall trust score under the update rules: '
    f'{trust_history[-1] - trust_history[0]:+.3f}\n'
)

print('EXPERT-PROFILE SUMMARIES:')
for area, performance in analytics['expert_performance'].items():
    expert = next(
        profile for profile in experts if profile.expertise_area == area
    )
    print(f'{area} ({expert.name}):')
    print(f'  Individual trust score: {performance["trust_level"]:.3f}')
    print(f'  Mean illustrative satisfaction: {performance["avg_satisfaction"]:.3f}')
    print(f'  Decision style: {expert.decision_style}')
    print()

print('INTERPRETATION:')
print('  Recommendation and feedback behavior is determined by explicit rules.')
print('  Trust and satisfaction are illustrative scores generated by those rules.')
print('  The notebook does not estimate human-only or AI-only outcomes.')
print('  The results do not establish that collaboration improves real-world decisions.\n')

print('PRACTICAL MODELING CONSIDERATIONS:')
print('  Validate equipment attributes and cost assumptions with relevant data.')
print('  Define explicit feasibility constraints before treating a configuration as implementable.')
print('  Include operating, maintenance, infrastructure, and lifecycle costs for a fuller assessment.')
print('  Evaluate feedback, trust, and satisfaction measures with appropriate evidence.')
print('  Retain human oversight for contextual judgment and strategic considerations.')